# 04: The training table

Build the table that the models learn from: one row per match, with the home team's form and the away team's form side by side, and the real goals as targets.

**Input:** `data/processed/matches.csv` and `data/processed/team_form.csv`.

**Output:** `data/processed/model_table.csv` (3,401 matches) and a time-based train/validation/test split.

**Contents**
1. Load the data
2. Join home and away features
3. Checks
4. Filter matches with thin history
5. Time-based split
6. Save and summary

## 1. Load the data

Create DuckDB views over the two CSV files. `types={'Season': 'VARCHAR'}` keeps season codes such as `2324` as text. If DuckDB read them as numbers, the comparisons between seasons used later in the split would not work as intended.

In [11]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.sql("""
    CREATE VIEW matches AS
    SELECT * FROM read_csv_auto('../data/processed/matches.csv', types={'Season': 'VARCHAR'})
""")
con.sql("""
    CREATE VIEW team_form AS
    SELECT * FROM read_csv_auto('../data/processed/team_form.csv', types={'Season': 'VARCHAR'})
""")

## 2. Join home and away features

Each match needs two lookups in `team_form`: the home team's row and the away team's row for that date. The same view is joined twice under different names, `h` for home and `a` for away, and the columns get the prefixes `h_` and `a_`.

The table holds four kinds of columns:
- **Keys:** season, date, home team, away team
- **Targets:** `home_goals` and `away_goals`, which the models predict
- **Features:** the `h_` and `a_` columns (points over the last 5, goals scored and conceded over the last 5 and 10, days of rest, prior matches), all calculated before kickoff
- **Benchmark only:** the Bet365 odds and the result `FTR`, used to evaluate the models but never as features

In [12]:
con.sql("""
    CREATE OR REPLACE VIEW model_table AS
    SELECT m.Season, m.Date, m.HomeTeam, m.AwayTeam,
           m.FTHG AS home_goals, m.FTAG AS away_goals, m.FTR,
           m.B365H, m.B365D, m.B365A,

           h.prior_matches        AS h_prior,
           h.points_last5         AS h_points_last5,
           h.goals_for_last5      AS h_gf5,
           h.goals_against_last5  AS h_ga5,
           h.goals_for_last10     AS h_gf10,
           h.goals_against_last10 AS h_ga10,
           h.days_rest            AS h_rest,

           a.prior_matches        AS a_prior,
           a.points_last5         AS a_points_last5,
           a.goals_for_last5      AS a_gf5,
           a.goals_against_last5  AS a_ga5,
           a.goals_for_last10     AS a_gf10,
           a.goals_against_last10 AS a_ga10,
           a.days_rest            AS a_rest
    FROM matches m
    JOIN team_form h ON h.Team = m.HomeTeam AND h.Date = m.Date AND h.Venue = 'H'
    JOIN team_form a ON a.Team = m.AwayTeam AND a.Date = m.Date AND a.Venue = 'A'
""")

## 3. Checks

### Row count

The joins must neither lose nor duplicate matches, so the table must have as many rows as `matches`.

In [13]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM model_table) AS model_rows,
           (SELECT COUNT(*) FROM matches) AS match_rows
""").df()

,model_rows,match_rows
0,3520,3520


**Result:** 3,520 rows in both.

### Spot check

Pick one match and compare its features with the `team_form` row for the same team and date.

In [14]:
df = con.sql("SELECT * FROM model_table ORDER BY Date").df()
row = df[df["HomeTeam"] == "Arsenal"].iloc[20]
print(row[["Date", "HomeTeam", "AwayTeam", "h_points_last5", "h_gf5"]])

con.sql(f"""
    SELECT Date, Team, points_last5, goals_for_last5
    FROM team_form
    WHERE Team = 'Arsenal' AND Date = '{row['Date'].date()}'
""").df()

Date              2018-08-25 00:00:00
HomeTeam                      Arsenal
AwayTeam                     West Ham
h_points_last5                      6
h_gf5                             1.8
Name: 400, dtype: object


,Date,Team,points_last5,goals_for_last5
0,2018-08-25,Arsenal,6,1.8


**Result:** for Arsenal vs West Ham on 25 Aug 2018, the model table shows 6 points and 1.8 goals scored per match over Arsenal's last five, and `team_form` shows exactly the same values.

## 4. Filter matches with thin history

Features built on fewer than 5 earlier matches are unreliable, and the models cannot handle missing values. Matches where either team has fewer than 5 prior matches are dropped.

In [15]:
n_before = len(df)
df = df[(df["h_prior"] >= 5) & (df["a_prior"] >= 5)].reset_index(drop=True)
print(n_before, "->", len(df), "rows")


3520 -> 3401 rows


**Result:** 3,520 rows become 3,401, so 119 matches were removed. They are the opening five rounds of 2017/18 (50 matches, when every club starts with no history) and the first five matches of each club that had never appeared in the data before: three clubs in each of 2018/19 and 2019/20 (15 matches each), one club per season from 2020/21 to 2025/26 (5 each), and Coventry and Hull in 2026/27 (9 matches).

## 5. Time-based split

Split by season, never at random:
- **Train:** seasons up to 2023/24
- **Validation:** 2024/25, for comparing models and choosing settings
- **Test:** 2025/26, kept untouched until the final evaluation
- **Held out:** the in-progress 2026/27 season, for a live demo later

Random shuffling would let a model learn from matches played after the ones it is predicting, which never happens in real use. Season codes compare correctly as text (`"2324" <= "2324"`), which is why `Season` was loaded as text.

In [16]:
train = df[df["Season"] <= "2324"]
valid = df[df["Season"] == "2425"]
test  = df[df["Season"] == "2526"]

print(len(train), len(valid), len(test))

2560 375 375


**Result:** 2,560 training, 375 validation and 375 test matches, plus 91 matches from 2026/27. Validation and test have 375 matches instead of 380 because each lost the first five matches of one newly promoted club (Ipswich in 2024/25, Sunderland in 2025/26).

## 6. Save the table

Save the filtered table to `data/processed/model_table.csv`. The notebooks that follow read it directly, so they do not need DuckDB.

In [17]:
df.to_csv("../data/processed/model_table.csv", index=False)

## Summary: the training table

**What this notebook built**
A one-row-per-match table (`model_table`) for predicting the goals each team scores. It combines match results with the rolling-form features from `03_sql`, so each match has the home team's form and the away team's form side by side.

**How it was built**
- Loaded `matches.csv` and `team_form.csv` into DuckDB
- Joined `team_form` twice (home team, away team) and prefixed the columns `h_` and `a_`
- Kept the real goals (`home_goals`, `away_goals`) as targets
- Kept the bookmaker odds for benchmarking only, never as features

**Features (all calculated before kickoff)**
- Points over the last 5 matches
- Average goals scored and conceded over the last 5 and 10 matches
- Days of rest
- `prior_matches`, used to filter out thin history

**Checks**
- `model_table` has 3,520 rows and `matches` has 3,520, so the joins lost and duplicated nothing
- A spot check of one match agrees with `team_form`
- The data covers 34 distinct teams, seasons 1718 to 2627

**Filtering and split**
- Dropped matches where either team had fewer than 5 prior matches: 119 rows removed, 3,401 remaining
- Time-based split with no shuffling:
  - Train: seasons up to 2023/24 (2,560 matches)
  - Validation: 2024/25 (375 matches), for comparing models
  - Test: 2025/26 (375 matches), used once at the end
- The partial 2026/27 season (91 matches) is held out for a live demo

**Leakage rules followed**
- Every feature uses only matches played before the one being predicted
- No shots, fouls, cards or odds from the current match are used as features
- The test season is not used for any decision before the final evaluation

**Limitations**
- The first matches of newly promoted clubs are excluded because of thin history
- Form carries across seasons, so late-season games feed early-season features
- No player, injury or lineup information is included

**Next:** `05_models.ipynb` builds the baseline and the Poisson regression models.